# Arrays & Dynamic Arrays: Zero to Hero

The structure everything else is built on, and the one whose performance comes from a property
Big-O cannot express: **the elements are next to each other in memory.**

> **Prerequisites:** [`complexity_zero_to_hero.ipynb`](complexity_zero_to_hero.ipynb) — this
> notebook uses its doubling experiment throughout, and picks up §1.5's amortised argument and
> §1.2's cache measurements where they left off.

***

## Why this notebook is different

An array is usually introduced as "a list with O(1) indexing" and then left alone. That misses
what actually makes it fast, and what it costs:

- **A growth factor is a design decision with a price tag.** §1.3 sweeps it: at 1.125× a dynamic
  array copies **8.24 elements per append**; at 2× it copies **1.31**; at 4×, **0.87** — and the
  4× version holds **162% more memory than it needs**. All of them are amortised O(1); the
  constant is the whole conversation.
- **Contiguity is worth more than the complexity class.** §3.1 measures the *same* traversal of
  the *same* Java array running **several times slower** in the wrong order — once the array is
  large enough to stop fitting in cache.
- **Saving memory can cost time.** §1.4: `array.array` stores integers in **8.5× less memory**
  than a Python list — and summing it is **2.6× slower**, because every access has to build a
  Python object that the list already had.
- **The famous accident.** §2.4 measures `list.insert(0, x)` in a loop at a clean **O(n²)** —
  the single most common way working code silently becomes quadratic.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Imports, JDK check, `dsa_toolkit` |
| **1. Theory from zero** | Contiguity and the address formula · what it costs · **the dynamic array from scratch, in both languages** · **memory: pointers vs machine ints, and the boxing tax** |
| **2. Techniques that fall out of contiguity** | **Prefix sums** · **difference arrays** · two pointers · sliding window · the quadratic accident |
| **3. The signature difficulty** | **Cache locality**, measured — row-major vs column-major, and array-of-structs vs struct-of-arrays |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice** | 8 exercises, ordered by difficulty |
| **6. Reading** | The chapters and papers behind each section |
| **Appendix** | Array-specific errors and a checklist |

## The one-paragraph summary

An array is a **contiguous block of equal-sized slots**, which is why `a[i]` is one multiply and
one add — the address is computable, not searchable. Everything good about arrays follows from
that (O(1) indexing, and hardware prefetching that makes sequential traversal far faster than the
cost model admits), and so does everything bad (inserting in the middle means shifting everything
after it, and growing means allocating a new block and copying). A **dynamic array** hides the
growing behind a **geometric growth factor**, which buys amortised O(1) append at the cost of some
wasted capacity — and choosing that factor is a real trade between copies and memory. The parts
that surprise people are that the *constant* matters more than the class here, and that two
implementations with identical Big-O can differ by a factor of ten because one of them respects
the cache and the other does not.

***
# Part 0 - Setup

Standard library only, plus a JDK for the Java half and `dsa_toolkit` (built in NB-00) for the
measurement and testing harness.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses.
# ---------------------------------------------------------------------------
import array
import math
import random
import sys
import time

from dsa_toolkit import (JavaError, StressFailure, check_invariant, cross_check,
                         edge_cases, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)
if not ok:
    print()
    print("The Java cells below will raise. Install a JDK and restart the kernel:")
    print("    winget install --id EclipseAdoptium.Temurin.21.JDK -e   (Windows)")

python 3.14.7
JDK available: True | javac 25.0.4.1


***
# Part 1 - Theory from zero

1. What an array *is*, and why indexing is O(1)
2. What contiguity costs
3. **The dynamic array, from scratch, in both languages**
4. **Memory: pointers, machine ints, and the boxing tax**

## 1.1 Contiguity, and the address formula

An array is a **contiguous block of memory divided into equal-sized slots**. That single sentence
contains every property arrays have.

Because the slots are equal-sized and adjacent, the address of element $i$ is *computed*:

$$ \text{addr}(a[i]) = \text{base} + i \times \text{sizeof(element)} $$

One multiply, one add. It does not depend on $i$, on the length, or on what is stored — which is
what "O(1) indexing" means, and it is why an array is the only structure that offers it without
conditions.

**The two requirements that buys it**, both of which cost you something later:

1. **Equal-sized slots.** So either the elements are all the same fixed-width type (Java's `int[]`),
   or the slots hold *pointers* to objects living elsewhere (Python's `list`, Java's `Integer[]`).
   §1.4 measures what that indirection costs.
2. **One contiguous block.** So the size must be known when you allocate, and growing means
   allocating a new block and copying. §1.3 is entirely about hiding that.

**And one property that is not in the complexity at all:** because the elements are adjacent, a
traversal reads memory in the order the hardware predicts, so the cache prefetcher can fetch ahead.
§3 measures this being worth more than most algorithmic improvements.

In [2]:
# ---------------------------------------------------------------------------
# The address formula, made literal.
# ---------------------------------------------------------------------------
ints = array.array("i", [10, 20, 30, 40, 50])       # 4-byte machine ints, contiguous
base, _ = ints.buffer_info()
size = ints.itemsize

print("array.array('i') -- a real contiguous block")
print("  base address : 0x%x" % base)
print("  element size : %d bytes\n" % size)
print("  %5s %20s %12s %8s" % ("i", "computed address", "= base + i*4", "value"))
print("  " + "-" * 50)
for i in range(len(ints)):
    print("  %5d %20s %12d %8d" % (i, "0x%x" % (base + i * size), i * size, ints[i]))

print()
print("Indexing does not search. It arithmetic. That is the entire trick, and it")
print("is why no other structure in this series gets O(1) access for free.")

array.array('i') -- a real contiguous block
  base address : 0x1b7aba4cad0
  element size : 4 bytes

      i     computed address = base + i*4    value
  --------------------------------------------------
      0        0x1b7aba4cad0            0       10
      1        0x1b7aba4cad4            4       20
      2        0x1b7aba4cad8            8       30
      3        0x1b7aba4cadc           12       40
      4        0x1b7aba4cae0           16       50

Indexing does not search. It arithmetic. That is the entire trick, and it
is why no other structure in this series gets O(1) access for free.


In [3]:
# ---------------------------------------------------------------------------
# Confirm it: indexing cost does not depend on the index or the length.
# ---------------------------------------------------------------------------
big = list(range(4_000_000))


def index_first(_):
    for _ in range(200_000):
        _v = big[0]


def index_last(_):
    n = len(big) - 1
    for _ in range(200_000):
        _v = big[n]


def index_random(idx):
    for i in idx:
        _v = big[i]


rng = random.Random(RANDOM_SEED)
positions = [rng.randrange(len(big)) for _ in range(200_000)]

print("200,000 index operations into a 4,000,000-element list:\n")
for label, fn, arg in [("always a[0]", index_first, None),
                       ("always a[-1] (by index)", index_last, None),
                       ("random positions", index_random, positions)]:
    best = min(measure_growth(fn, [1], setup=lambda _n, a=arg: a, repeats=5)[0]["seconds"]
               for _ in range(1))
    print("  %-26s %.4fs" % (label, best))

print()
print("The first two are identical: position does not matter. The third is slower,")
print("and NOT because indexing costs more -- the arithmetic is the same. It is the")
print("CACHE. Section 3 is about that gap.")

200,000 index operations into a 4,000,000-element list:

  always a[0]                0.0079s
  always a[-1] (by index)    0.0073s


  random positions           0.0544s

The first two are identical: position does not matter. The third is slower,
and NOT because indexing costs more -- the arithmetic is the same. It is the
CACHE. Section 3 is about that gap.


## 1.2 What contiguity costs

Every array operation is fast or slow for the same reason: whether it has to move anything.

| Operation | Cost | Why |
|---|---|---|
| `a[i]` read/write | $\Theta(1)$ | address arithmetic |
| append (amortised) | $\Theta(1)$ | usually a free slot; occasionally a resize (§1.3) |
| append (worst case) | $\Theta(n)$ | the resize itself copies everything |
| insert / delete at position $i$ | $\Theta(n-i)$ | everything after $i$ shifts |
| **insert / delete at the front** | $\Theta(n)$ | everything shifts |
| search (unsorted) | $\Theta(n)$ | no structure to exploit |
| search (sorted) | $\Theta(\log n)$ | binary search — NB-15 |

**The row that causes real incidents is the bold one.** `list.pop(0)` and `list.insert(0, x)` look
like O(1) operations and are O(n) each, so using either inside a loop is O(n²). §2.4 measures it.

In [4]:
# ---------------------------------------------------------------------------
# Shifting is the whole cost. Watch it depend on WHERE you insert.
# ---------------------------------------------------------------------------
N = 200_000
positions = {"front (0)": 0, "middle (n/2)": N // 2, "end (n)": N}

print("10,000 insertions into a %s-element list, at a fixed position:\n"
      % "{:,}".format(N))
print("  %-16s %12s" % ("position", "seconds"))
print("  " + "-" * 30)
for label, pos in positions.items():
    best = float("inf")
    for _ in range(3):
        data = list(range(N))
        p = min(pos, len(data))
        t0 = time.perf_counter()
        for _ in range(10_000):
            data.insert(p, 0)
        best = min(best, time.perf_counter() - t0)
    print("  %-16s %12.4f" % (label, best))

print()
print("Inserting at the end is effectively free; at the front it copies the whole")
print("array every time. The cost is exactly the number of elements to the right.")
print()
print("If you need to add at the front, you do not want an array -- you want a")
print("deque (NB-05) or a linked list (NB-04), and NB-04 measures when each wins.")

10,000 insertions into a 200,000-element list, at a fixed position:

  position              seconds
  ------------------------------


  front (0)              1.0845


  middle (n/2)           0.4870
  end (n)                0.0229

Inserting at the end is effectively free; at the front it copies the whole
array every time. The cost is exactly the number of elements to the right.

If you need to add at the front, you do not want an array -- you want a
deque (NB-05) or a linked list (NB-04), and NB-04 measures when each wins.


## 1.3 The dynamic array, from scratch

A fixed array cannot grow. A **dynamic array** — Python's `list`, Java's `ArrayList`, C++'s
`vector` — hides that behind three fields: a buffer, a **capacity** (how many slots exist), and a
**size** (how many are used).

```
capacity = 8
size     = 5
buffer  [ a | b | c | d | e | · | · | · ]
                             ^ size          ^ capacity
```

**The invariant:** `0 <= size <= capacity`, and slots `[0, size)` hold real elements. Append
writes at `size` and increments it. When `size == capacity`, allocate a bigger buffer, copy, and
carry on.

**The whole design question is how much bigger.** Grow by a constant and appending becomes
$\Theta(n)$ each and $\Theta(n^2)$ overall — NB-00 §1.5 measured that. Grow by a *factor* and it
is amortised $\Theta(1)$, provable three ways:

- **Aggregate.** With factor 2, $n$ appends copy $1 + 2 + 4 + \dots < 2n$ elements total.
- **Accounting.** Charge 3 units per append: 1 to store, 2 banked. A resize spends exactly the
  bank. The bank never goes negative, so the amortised cost is the charge — $O(1)$.
- **Potential.** $\Phi = 2\cdot\text{size} - \text{capacity}$. Cheap appends raise it; a resize
  spends it; actual + $\Delta\Phi$ is constant.

The cell below implements it with the invariant asserted after **every** operation, and counts the
copies so the argument is checked rather than believed.

In [5]:
# ---------------------------------------------------------------------------
# A dynamic array, with its invariant checked after every single operation.
# ---------------------------------------------------------------------------
class DynamicArray:
    """A growable array over a fixed buffer. Counts copies so the amortised
    argument can be verified rather than asserted."""

    def __init__(self, factor=2.0):
        self._buf = [None]          # capacity 1
        self._size = 0
        self._factor = factor
        self.copies = 0             # elements physically moved
        self.resizes = 0

    # -- the invariant that defines the structure --------------------------
    @staticmethod
    def invariant(da):
        if not (0 <= da._size <= len(da._buf)):
            return "size %d outside [0, capacity %d]" % (da._size, len(da._buf))
        if len(da._buf) < 1:
            return "capacity fell to zero"
        return True

    def _grow(self):
        new_cap = max(len(self._buf) + 1, int(len(self._buf) * self._factor))
        bigger = [None] * new_cap
        for i in range(self._size):          # the copy the whole argument is about
            bigger[i] = self._buf[i]
        self.copies += self._size
        self.resizes += 1
        self._buf = bigger

    def append(self, value):
        if self._size == len(self._buf):
            self._grow()
        self._buf[self._size] = value
        self._size += 1

    def __getitem__(self, i):
        if not 0 <= i < self._size:
            raise IndexError(i)
        return self._buf[i]

    def __setitem__(self, i, v):
        if not 0 <= i < self._size:
            raise IndexError(i)
        self._buf[i] = v

    def __len__(self):
        return self._size

    def pop(self):
        if self._size == 0:
            raise IndexError("pop from empty array")
        self._size -= 1
        return self._buf[self._size]

    @property
    def capacity(self):
        return len(self._buf)

    def __repr__(self):
        return "DynamicArray(size=%d, capacity=%d)" % (self._size, self.capacity)


# Stress it: random operations, invariant checked after each one, and the whole
# structure compared against a plain list at every step.
rng = random.Random(RANDOM_SEED)
da, reference = DynamicArray(), []
for step in range(20_000):
    op = rng.random()
    if op < 0.75 or not reference:
        v = rng.randrange(1000)
        da.append(v); reference.append(v)
    else:
        assert da.pop() == reference.pop()
    check_invariant(da, DynamicArray.invariant, "dynamic array", "step %d" % step)
    assert len(da) == len(reference)

assert [da[i] for i in range(len(da))] == reference
print("20,000 random append/pop operations")
print("  invariant held after every one")
print("  contents identical to a plain list at every step")
print("  final:", da)

20,000 random append/pop operations
  invariant held after every one
  contents identical to a plain list at every step
  final: DynamicArray(size=9732, capacity=16384)


In [6]:
# ---------------------------------------------------------------------------
# The growth factor is a trade-off. Measure both sides of it.
# ---------------------------------------------------------------------------
N_APPEND = 100_000

print("%s appends, starting from capacity 1:\n" % "{:,}".format(N_APPEND))
print("  %8s %10s %13s %15s %12s %10s"
      % ("factor", "resizes", "copies", "copies/append", "final cap", "waste"))
print("  " + "-" * 74)
for factor in [1.125, 1.25, 1.5, 2.0, 4.0]:
    d = DynamicArray(factor)
    for i in range(N_APPEND):
        d.append(i)
    waste = (d.capacity - N_APPEND) / N_APPEND
    print("  %8.3f %10s %13s %15.2f %12s %9.1f%%"
          % (factor, "{:,}".format(d.resizes), "{:,}".format(d.copies),
             d.copies / N_APPEND, "{:,}".format(d.capacity), 100 * waste))

print()
print("Read it as a trade, because that is what it is:")
print("  - a SMALL factor copies more (8.24 elements per append at 1.125x) and")
print("    wastes almost no memory;")
print("  - a LARGE factor barely copies (0.87 per append at 4x) and can leave the")
print("    buffer more than half empty.")
print()
print("Every row is amortised O(1) -- copies per append is a constant in all of")
print("them. The constant is what you are choosing, and the two real")
print("implementations sit at opposite ends: CPython's list uses about 1.125x,")
print("Java's ArrayList uses 1.5x.")

100,000 appends, starting from capacity 1:

    factor    resizes        copies   copies/append    final cap      waste
  --------------------------------------------------------------------------
     1.125         91       823,529            8.24      102,908       2.9%


     1.250         51       482,072            4.82      120,501      20.5%
     1.500         29       276,521            2.77      138,255      38.3%
     2.000         17       131,071            1.31      131,072      31.1%


     4.000          9        87,381            0.87      262,144     162.1%

Read it as a trade, because that is what it is:
  - a SMALL factor copies more (8.24 elements per append at 1.125x) and
    wastes almost no memory;
  - a LARGE factor barely copies (0.87 per append at 4x) and can leave the
    buffer more than half empty.

Every row is amortised O(1) -- copies per append is a constant in all of
them. The constant is what you are choosing, and the two real
implementations sit at opposite ends: CPython's list uses about 1.125x,
Java's ArrayList uses 1.5x.


In [7]:
# ---------------------------------------------------------------------------
# And it really is O(n) overall -- the amortised claim, measured.
# ---------------------------------------------------------------------------
def append_n(n, factor=2.0):
    d = DynamicArray(factor)
    for i in range(n):
        d.append(i)
    return d


growth_table(measure_growth(lambda n: append_n(n, 2.0),
                            [100_000, 200_000, 400_000, 800_000], repeats=3),
             claim="O(n)")
print()
print("The ratio sits at about 2, so n appends cost O(n) in total -- O(1) each,")
print("amortised -- despite individual appends occasionally costing O(n). That is")
print("the whole point of the word.")
print()
print("If the harness reports NOT SEPARABLE here, that is NB-00 3.2 doing its job:")
print("over an 8x range of sizes, timing noise is larger than the log n term, so")
print("O(n) and O(n log n) fit about equally well. The RATIO column is the")
print("unambiguous evidence, and it says 2.")

         n        seconds      ratio
------------------------------------
   100,000       0.026034          -
   200,000       0.052524       2.02
   400,000       0.106461       2.03
   800,000       0.218288       2.05

best fit: O(n) (relative error 0.018); next: O(n log n) (0.045)
claimed O(n) -> measurement MATCHES the claim

The ratio sits at about 2, so n appends cost O(n) in total -- O(1) each,
amortised -- despite individual appends occasionally costing O(n). That is
the whole point of the word.

If the harness reports NOT SEPARABLE here, that is NB-00 3.2 doing its job:
over an 8x range of sizes, timing noise is larger than the log n term, so
O(n) and O(n log n) fit about equally well. The RATIO column is the
unambiguous evidence, and it says 2.


In [8]:
# ---------------------------------------------------------------------------
# The same structure in Java, cross-checked against the Python one.
# ---------------------------------------------------------------------------
JAVA_DYNAMIC = """
public class DynGrow {
    private int[] buf = new int[1];
    private int size = 0;
    int copies = 0, resizes = 0;

    void append(int v) {
        if (size == buf.length) {
            int newCap = Math.max(buf.length + 1, buf.length * 2);
            int[] bigger = new int[newCap];
            System.arraycopy(buf, 0, bigger, 0, size);   // the resize copy
            copies += size; resizes++; buf = bigger;
        }
        buf[size++] = v;
    }

    int capacity() { return buf.length; }

    public static void main(String[] args) {
        java.util.Scanner sc = new java.util.Scanner(System.in);
        int n = sc.nextInt();
        DynGrow d = new DynGrow();
        for (int i = 0; i < n; i++) d.append(i);
        System.out.println(d.resizes + " " + d.copies + " " + d.capacity());
    }
}
"""


def python_growth_signature(n):
    """The same three numbers, from the Python implementation."""
    d = DynamicArray(2.0)
    for i in range(n):
        d.append(i)
    return "%d %d %d" % (d.resizes, d.copies, d.capacity)


checked = cross_check(python_growth_signature, JAVA_DYNAMIC,
                      lambda r: r.randrange(0, 5000),
                      lambda n: "%d\n" % n,
                      n=25, label="dynamic array growth")
print("%d random sizes: Python and Java agree exactly on" % checked)
print("(resizes, elements copied, final capacity).")
print()
print("That is a stronger statement than 'both work'. The two implementations")
print("perform the SAME sequence of allocations, so the amortised argument")
print("verified above applies to the Java one unchanged.")

25 random sizes: Python and Java agree exactly on
(resizes, elements copied, final capacity).

That is a stronger statement than 'both work'. The two implementations
perform the SAME sequence of allocations, so the amortised argument
verified above applies to the Java one unchanged.


## 1.4 Memory: pointers, machine ints, and the boxing tax

§1.1 said the slots must be equal-sized, and offered two ways to arrange that. This is where the
choice shows up.

| | Slot holds | Element cost |
|---|---|---|
| Java `int[]` | the value itself, 4 bytes | 4 bytes, contiguous |
| Java `Integer[]` | an 8-byte reference | 8 bytes **+ ~16-byte object elsewhere** |
| Python `list` | an 8-byte reference | 8 bytes **+ ~28-byte int object elsewhere** |
| Python `array.array('i')` | the value itself, 4 bytes | 4 bytes, contiguous |

Python's `list` has no choice: it holds references to objects, always. `array.array` is the
contiguous alternative, and the comparison is more interesting than "use the smaller one".

In [9]:
# ---------------------------------------------------------------------------
# Measure the memory honestly: a list's ints are separate objects.
# ---------------------------------------------------------------------------
M = 2_000_000
py_list = list(range(M))
arr_i = array.array("i", range(M))


def deep_size(lst):
    """getsizeof(list) is only the pointer array. Add the int objects it points to.
    CPython interns small ints (-5..256), so those are shared, not per-element."""
    return sys.getsizeof(lst) + sum(sys.getsizeof(v) for v in lst if not -5 <= v <= 256)


list_bytes, arr_bytes = deep_size(py_list), sys.getsizeof(arr_i)

print("%s integers, 0..n-1\n" % "{:,}".format(M))
print("  %-24s %16s %14s" % ("representation", "total bytes", "per element"))
print("  " + "-" * 58)
print("  %-24s %16s %14.1f" % ("list of int objects", "{:,}".format(list_bytes),
                               list_bytes / M))
print("  %-24s %16s %14.1f" % ("array.array('i')", "{:,}".format(arr_bytes),
                               arr_bytes / M))
print("\n  the list costs %.1fx the memory" % (list_bytes / arr_bytes))
print()
print("A naive sys.getsizeof(list) reports about 8 bytes per element and makes")
print("this look like a tie. It is only counting the pointer array; the int")
print("objects it points at are the other 28 bytes each.")

2,000,000 integers, 0..n-1

  representation                total bytes    per element
  ----------------------------------------------------------
  list of int objects            71,992,860           36.0
  array.array('i')                8,470,312            4.2

  the list costs 8.5x the memory

A naive sys.getsizeof(list) reports about 8 bytes per element and makes
this look like a tie. It is only counting the pointer array; the int
objects it points at are the other 28 bytes each.


In [10]:
# ---------------------------------------------------------------------------
# ... and now the part that is NOT a free win.
# ---------------------------------------------------------------------------
def best_of(fn, arg, repeats=5):
    b = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(arg)
        b = min(b, time.perf_counter() - t0)
    return b


t_list, t_arr = best_of(sum, py_list), best_of(sum, arr_i)
print("summing the same %s integers:\n" % "{:,}".format(M))
print("  list        %.4fs" % t_list)
print("  array.array %.4fs   -> %.1fx SLOWER" % (t_arr, t_arr / t_list))
print()
print("The compact representation is the slower one to iterate, in Python.")
print()
print("Why: array.array stores raw machine ints, so every element you touch has to")
print("be BOXED into a fresh Python int object before Python can do anything with")
print("it. The list already holds objects, so iterating it just follows pointers.")
print()
print("So the trade is real and it goes both ways:")
print("  array.array   8.5x less memory, and slower elementwise access from Python")
print("  list          fast Python-level access, and 8.5x the memory")
print()
print("array.array wins outright when the data is large and you hand it to")
print("something that reads the buffer directly -- a file write, a socket, numpy,")
print("or a C library. It loses when you loop over it in Python.")

summing the same 2,000,000 integers:

  list        0.0135s
  array.array 0.0302s   -> 2.2x SLOWER

The compact representation is the slower one to iterate, in Python.

Why: array.array stores raw machine ints, so every element you touch has to
be BOXED into a fresh Python int object before Python can do anything with
it. The list already holds objects, so iterating it just follows pointers.

So the trade is real and it goes both ways:
  array.array   8.5x less memory, and slower elementwise access from Python
  list          fast Python-level access, and 8.5x the memory

array.array wins outright when the data is large and you hand it to
something that reads the buffer directly -- a file write, a socket, numpy,
or a C library. It loses when you loop over it in Python.


In [11]:
# ---------------------------------------------------------------------------
# Java has the same choice, and there the compact one wins on both counts.
# ---------------------------------------------------------------------------
JAVA_BOXING = """
public class Boxing {
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);
        int[] prim = new int[n];
        Integer[] boxed = new Integer[n];
        for (int i = 0; i < n; i++) { prim[i] = i; boxed[i] = i; }

        long s = 0;
        for (int w = 0; w < 3; w++) {            // JIT warm-up on both paths
            for (int i = 0; i < n; i++) s += prim[i];
            for (int i = 0; i < n; i++) s += boxed[i];
        }
        long t0 = System.nanoTime();
        for (int i = 0; i < n; i++) s += prim[i];
        double tPrim = (System.nanoTime() - t0) / 1e9;

        t0 = System.nanoTime();
        for (int i = 0; i < n; i++) s += boxed[i];
        double tBoxed = (System.nanoTime() - t0) / 1e9;

        System.out.printf("%.6f %.6f %d%n", tPrim, tBoxed, s);
    }
}
"""

out = run_java(JAVA_BOXING, args=[20_000_000]).split()
t_prim, t_boxed = float(out[0]), float(out[1])

print("summing 20,000,000 elements in Java:\n")
print("  int[]      %.4fs   4 bytes/element, contiguous" % t_prim)
print("  Integer[]  %.4fs   8-byte reference + a ~16-byte object each" % t_boxed)
print("  Integer[] is %.1fx slower\n" % (t_boxed / t_prim))
print("Unlike Python's array.array, there is no boxing tax on READING an int[] --")
print("the value is right there. Integer[] pays twice: more memory, AND a pointer")
print("dereference per element that lands wherever the allocator put the object.")
print()
print("This is why `List<Integer>` is a performance trap in hot Java code, and why")
print("libraries that care (Trove, fastutil, and the JDK's own IntStream) all offer")
print("primitive-specialised collections.")

summing 20,000,000 elements in Java:

  int[]      0.0110s   4 bytes/element, contiguous
  Integer[]  0.0310s   8-byte reference + a ~16-byte object each
  Integer[] is 2.8x slower

Unlike Python's array.array, there is no boxing tax on READING an int[] --
the value is right there. Integer[] pays twice: more memory, AND a pointer
dereference per element that lands wherever the allocator put the object.

This is why `List<Integer>` is a performance trap in hot Java code, and why
libraries that care (Trove, fastutil, and the JDK's own IntStream) all offer
primitive-specialised collections.


***
# Part 2 - The techniques that fall out of contiguity

Prefix sums, difference arrays, two pointers and sliding windows are usually taught as a list of
tricks to memorise. They are not tricks. Each one is a direct consequence of the two things §1.1
established: **you can compute any index in O(1)**, and **walking forward is cheap**.

Learn them that way and you can re-derive them; memorise them and you will not recognise the next
problem that wants one.

## 2.1 Prefix sums: pay O(n) once, answer range queries in O(1)

**The problem.** Given a fixed array, answer many queries of the form "what is the sum of
`a[lo:hi]`?" Naively each query is $\Theta(n)$, so $q$ queries cost $\Theta(qn)$.

**The derivation.** Define $P[k] = a[0] + a[1] + \dots + a[k-1]$, with $P[0] = 0$. Then

$$ \sum_{i=lo}^{hi-1} a[i] = P[hi] - P[lo] $$

because everything before `lo` appears in both terms and cancels. Building $P$ is one pass,
$\Theta(n)$; every query afterwards is one subtraction, $\Theta(1)$.

**Why an array makes this work:** $P[hi]$ and $P[lo]$ are both O(1) to reach. On a linked list the
same identity is true and useless, because getting to $P[hi]$ costs $\Theta(hi)$.

The `+1` offset and the half-open interval are where the bugs live, so the cell below checks the
implementation against a brute-force reference over thousands of random ranges rather than one
worked example.

In [12]:
# ---------------------------------------------------------------------------
# Prefix sums, with the boundary convention made explicit.
# ---------------------------------------------------------------------------
def build_prefix(a):
    """P[k] = sum of a[0:k].  len(P) == len(a) + 1, and P[0] == 0."""
    p = [0] * (len(a) + 1)
    for i, v in enumerate(a):
        p[i + 1] = p[i] + v
    return p


def range_sum(p, lo, hi):
    """Sum of a[lo:hi] -- half-open, like every other Python slice."""
    return p[hi] - p[lo]


demo = [3, 1, 4, 1, 5, 9, 2, 6]
P = build_prefix(demo)
print("a =", demo)
print("P =", P, "   <- one longer, and starts at 0\n")
print("  %-14s %10s %10s" % ("query", "P[hi]-P[lo]", "sum(a[lo:hi])"))
print("  " + "-" * 38)
for lo, hi in [(0, 0), (0, 3), (2, 6), (5, 8), (0, 8)]:
    print("  a[%d:%d]%s %10d %10d" % (lo, hi, " " * (7 - len("%d%d" % (lo, hi))),
                                      range_sum(P, lo, hi), sum(demo[lo:hi])))


# Not one example -- thousands, against a brute-force reference, including the
# empty range and both endpoints.
def impl(case):
    a, lo, hi = case
    return range_sum(build_prefix(a), lo, hi)


def reference(case):
    a, lo, hi = case
    return sum(a[lo:hi])


def gen(r):
    a = [r.randrange(-50, 50) for _ in range(r.randrange(0, 30))]
    lo = r.randrange(0, len(a) + 1)
    hi = r.randrange(lo, len(a) + 1)
    return (a, lo, hi)


extra = [([], 0, 0), ([5], 0, 0), ([5], 0, 1), ([1, 2, 3], 3, 3), ([1, 2, 3], 0, 3)]
n = stress(impl, reference, gen, n=5000, extra=extra, label="range_sum")
print("\n%s cases (including the empty range and both endpoints): all agree."
      % "{:,}".format(n))

a = [3, 1, 4, 1, 5, 9, 2, 6]
P = [0, 3, 4, 8, 9, 14, 23, 25, 31]    <- one longer, and starts at 0

  query          P[hi]-P[lo] sum(a[lo:hi])
  --------------------------------------
  a[0:0]               0          0
  a[0:3]               8          8
  a[2:6]              19         19
  a[5:8]              17         17
  a[0:8]              31         31

5,005 cases (including the empty range and both endpoints): all agree.


## 2.2 Difference arrays: the same idea, run backwards

Prefix sums make **range queries** cheap on a fixed array. The mirror image makes **range updates**
cheap when you do not need to read until the end.

**The problem.** Apply many updates of the form "add $\delta$ to every element in `a[lo:hi]`", then
read the final array. Naively each update is $\Theta(hi-lo)$, so $u$ full-range updates cost
$\Theta(un)$.

**The derivation.** Store the *differences* instead. In $D$, record `D[lo] += δ` and `D[hi] -= δ`
— two writes, whatever the range's length. The running sum of $D$ reconstructs the array, because
the $+\delta$ switches on at `lo` and the $-\delta$ switches it off at `hi`.

Each update becomes $\Theta(1)$, and one final $\Theta(n)$ pass recovers the answer:
$\Theta(u + n)$ instead of $\Theta(un)$.

This is exactly the relationship between differentiation and integration, and it is worth
noticing that the prefix-sum pass *is* the reconstruction.

In [13]:
# ---------------------------------------------------------------------------
# Difference array: O(1) per range update, one O(n) pass at the end.
# ---------------------------------------------------------------------------
def apply_updates_naive(n, updates):
    a = [0] * n
    for lo, hi, delta in updates:
        for i in range(lo, hi):          # touches the whole range
            a[i] += delta
    return a


def apply_updates_difference(n, updates):
    diff = [0] * (n + 1)                 # the +1 makes hi == n safe
    for lo, hi, delta in updates:
        diff[lo] += delta                # switch the delta ON at lo
        diff[hi] -= delta                # switch it OFF at hi
    out, running = [0] * n, 0
    for i in range(n):                   # one prefix-sum pass reconstructs it
        running += diff[i]
        out[i] = running
    return out


ups = [(0, 5, 10), (2, 8, 3), (6, 10, -4)]
print("array of 10 zeros, updates:", ups)
print("  naive      ", apply_updates_naive(10, ups))
print("  difference ", apply_updates_difference(10, ups))

n = stress(lambda c: apply_updates_difference(*c),
           lambda c: apply_updates_naive(*c),
           lambda r: (lambda size: (size, [(lambda lo: (lo, r.randrange(lo, size) + 1,
                                                        r.randrange(-9, 9)))(r.randrange(0, size))
                                           for _ in range(r.randrange(0, 8))]))(r.randrange(1, 25)),
           n=3000, extra=[(1, []), (1, [(0, 1, 5)]), (3, [(0, 3, 1), (0, 3, -1)])],
           label="difference array")
print("\n%s random update sets: identical to the naive version every time."
      % "{:,}".format(n))

array of 10 zeros, updates: [(0, 5, 10), (2, 8, 3), (6, 10, -4)]
  naive       [10, 10, 13, 13, 13, 3, -1, -1, -4, -4]
  difference  [10, 10, 13, 13, 13, 3, -1, -1, -4, -4]

3,003 random update sets: identical to the naive version every time.


In [14]:
# ---------------------------------------------------------------------------
# The saving is in the NUMBER OF UPDATES, so sweep that, holding n fixed.
# ---------------------------------------------------------------------------
N_FIXED = 50_000


def full_range_updates(u):
    return [(0, N_FIXED, 1)] * u


print("array of %s elements; sweeping how many range updates are applied:\n"
      % "{:,}".format(N_FIXED))
print("  %10s %14s %16s %12s" % ("updates", "naive (s)", "difference (s)", "speed-up"))
print("  " + "-" * 58)
for u in [25, 50, 100, 200, 400]:
    ups = full_range_updates(u)
    tn = min(measure_growth(lambda _x: apply_updates_naive(N_FIXED, ups), [1],
                            repeats=3)[0]["seconds"] for _ in range(1))
    td = min(measure_growth(lambda _x: apply_updates_difference(N_FIXED, ups), [1],
                            repeats=3)[0]["seconds"] for _ in range(1))
    print("  %10d %14.5f %16.5f %11.0fx" % (u, tn, td, tn / td))

print()
print("Confirm the classes, in u (the number of updates), with n held fixed:\n")
growth_table(measure_growth(lambda u: apply_updates_naive(N_FIXED, full_range_updates(u)),
                            [50, 100, 200, 400], repeats=3), claim="O(n)")
print()
growth_table(measure_growth(lambda u: apply_updates_difference(N_FIXED, full_range_updates(u)),
                            [50, 100, 200, 400], repeats=3), claim="O(1)")

print()
print("Naive grows linearly with the number of updates; the difference array is")
print("flat in u. (Strictly it is O(u + n), and with n pinned the constant O(n)")
print("reconstruction pass dominates -- which is why the ratio sits near 1.)")

array of 50,000 elements; sweeping how many range updates are applied:

     updates      naive (s)   difference (s)     speed-up
  ----------------------------------------------------------


          25        0.07436          0.00263          28x


          50        0.15721          0.00262          60x


         100        0.29416          0.00265         111x


         200        0.59770          0.00272         219x


         400        1.32978          0.00322         413x

Confirm the classes, in u (the number of updates), with n held fixed:



         n        seconds      ratio
------------------------------------
        50       0.135564          -
       100       0.285652       2.11
       200       0.554475       1.94
       400       1.335060       2.41

best fit: O(n) (relative error 0.079); next: O(n log n) (0.107)
claimed O(n) -> measurement MATCHES the claim

         n        seconds      ratio
------------------------------------
        50       0.002615          -
       100       0.002677       1.02
       200       0.002752       1.03
       400       0.003391       1.23

best fit: O(log n) (relative error 0.090); next: O(1) (0.099)
NOT SEPARABLE: O(log n) and O(1) fit these timings about equally well.
  Read the ratio column instead, and widen the range of sizes or
  count operations rather than timing them (NB-00 1.7) if you need
  to settle it.
claimed O(1) -> CONSISTENT with the measurement, which cannot
  distinguish it from O(log n) here.

Naive grows linearly with the number of updates; the differenc

## 2.3 Two pointers and sliding windows

Both are the same observation: **when the array has an order you can exploit, you can move an
index forward and never move it back.** That makes an apparently $\Theta(n^2)$ search $\Theta(n)$,
and it works only because advancing an index in an array is free.

**Two pointers (opposite ends).** On a *sorted* array, to find a pair summing to a target: start at
both ends. If the sum is too big the right element is too big for *any* remaining left, so `hi--`
discards a whole column of the search space. Too small and `lo++` does the same for a row. Each
step eliminates one candidate permanently, so it terminates in at most $n$ steps.

**Sliding window.** For "longest/shortest subarray satisfying P", where extending the window can
only push P one way and shrinking the other: advance `hi` to include, advance `lo` to restore the
condition. **Each index moves forward at most $n$ times in total**, so the nested-looking loop is
$\Theta(n)$, not $\Theta(n^2)$ — and that amortised argument is the one people get wrong when
asked why.

In [15]:
# ---------------------------------------------------------------------------
# Two pointers on a sorted array, checked against the brute-force O(n^2).
# ---------------------------------------------------------------------------
def two_sum_sorted(case):
    """Indices of a pair summing to target, or None. Sorted input assumed."""
    a, target = case
    lo, hi = 0, len(a) - 1
    while lo < hi:
        s = a[lo] + a[hi]
        if s == target:
            return (lo, hi)
        if s < target:
            lo += 1          # a[lo] too small for ANY remaining partner
        else:
            hi -= 1          # a[hi] too big for ANY remaining partner
    return None


def two_sum_bruteforce(case):
    a, target = case
    for i in range(len(a)):
        for j in range(i + 1, len(a)):
            if a[i] + a[j] == target:
                return (i, j)
    return None


# The pair found may differ between the two, so compare the SUM, not the indices.
def as_sum(fn):
    def wrapped(case):
        r = fn(case)
        return None if r is None else case[0][r[0]] + case[0][r[1]]
    return wrapped


def gen_sorted(r):
    a = sorted(r.randrange(-30, 30) for _ in range(r.randrange(0, 14)))
    return (a, r.randrange(-60, 60))


n = stress(as_sum(two_sum_sorted), as_sum(two_sum_bruteforce), gen_sorted,
           n=4000, extra=[(([], 0)), (([1], 1)), (([1, 1], 2)), (([-2, -1, 3], 1))],
           label="two_sum_sorted")
print("two pointers: %s cases agree with brute force" % "{:,}".format(n))

growth_table(measure_growth(
    lambda n_: two_sum_sorted((list(range(n_)), -1)),      # worst case: no answer
    [200_000, 400_000, 800_000, 1_600_000], repeats=3), claim="O(n)")
print()
print("Worst case (no pair exists) still walks the array once, not n^2 times.")

two pointers: 4,004 cases agree with brute force


         n        seconds      ratio
------------------------------------
   200,000       0.037248          -
   400,000       0.074961       2.01
   800,000       0.148005       1.97
 1,600,000       0.300784       2.03

best fit: O(n) (relative error 0.006); next: O(n log n) (0.057)
claimed O(n) -> measurement MATCHES the claim

Worst case (no pair exists) still walks the array once, not n^2 times.


In [16]:
# ---------------------------------------------------------------------------
# Sliding window, and the amortised argument that makes it linear.
# ---------------------------------------------------------------------------
def longest_window_sum_at_most(case):
    """Longest subarray of NON-NEGATIVE values with sum <= limit."""
    a, limit = case
    best, total, lo = 0, 0, 0
    for hi in range(len(a)):
        total += a[hi]
        while total > limit and lo <= hi:
            total -= a[lo]          # lo only ever moves FORWARD
            lo += 1
        best = max(best, hi - lo + 1)
    return best


def longest_window_bruteforce(case):
    a, limit = case
    best = 0
    for i in range(len(a)):
        s = 0
        for j in range(i, len(a)):
            s += a[j]
            if s <= limit:
                best = max(best, j - i + 1)
            else:
                break
    return best


n = stress(longest_window_sum_at_most, longest_window_bruteforce,
           lambda r: ([r.randrange(0, 12) for _ in range(r.randrange(0, 25))],
                      r.randrange(0, 40)),
           n=4000, extra=[(([], 5)), (([0, 0, 0], 0)), (([9], 3)), (([1, 1, 1], 100))],
           label="sliding window")
print("sliding window: %s cases agree with the O(n^2) reference" % "{:,}".format(n))

rng2 = random.Random(RANDOM_SEED)
growth_table(measure_growth(
    longest_window_sum_at_most,
    [100_000, 200_000, 400_000, 800_000], repeats=3,
    setup=lambda n_: ([rng2.randrange(0, 10) for _ in range(n_)], 25)), claim="O(n)")

print()
print("The loop LOOKS quadratic -- a while inside a for. It is not, and the reason")
print("is the amortised argument from NB-00 1.5: `lo` never decreases, so across")
print("the whole run it advances at most n times in total. Two counters, each")
print("moving forward at most n steps, is 2n steps, which is O(n).")

sliding window: 4,004 cases agree with the O(n^2) reference


         n        seconds      ratio
------------------------------------
   100,000       0.025675          -
   200,000       0.047616       1.85
   400,000       0.101308       2.13
   800,000       0.195117       1.93

best fit: O(n) (relative error 0.030); next: O(n log n) (0.077)
claimed O(n) -> measurement MATCHES the claim

The loop LOOKS quadratic -- a while inside a for. It is not, and the reason
is the amortised argument from NB-00 1.5: `lo` never decreases, so across
the whole run it advances at most n times in total. Two counters, each
moving forward at most n steps, is 2n steps, which is O(n).


## 2.4 The quadratic accident

The single most common way working code becomes quadratic: an $\Theta(n)$ array operation used
inside a loop, where it looks like it costs nothing.

`list.insert(0, x)` and `list.pop(0)` each shift every element (§1.2). Put either in a loop and you
have written $\Theta(n^2)$ without typing a nested loop.

In [17]:
# ---------------------------------------------------------------------------
# Same task, three ways. Only one of them is linear.
# ---------------------------------------------------------------------------
from collections import deque


def build_reversed_insert(n):
    out = []
    for i in range(n):
        out.insert(0, i)          # O(n) each -> O(n^2) overall
    return out


def build_reversed_append(n):
    out = []
    for i in range(n):
        out.append(i)             # O(1) amortised
    out.reverse()                 # one O(n) pass
    return out


def build_reversed_deque(n):
    out = deque()
    for i in range(n):
        out.appendleft(i)         # O(1) -- a deque is not an array (NB-05)
    return list(out)


assert (build_reversed_insert(500) == build_reversed_append(500)
        == build_reversed_deque(500))
print("all three produce identical output.\n")

growth_table(measure_growth(build_reversed_insert, [5_000, 10_000, 20_000, 40_000],
                            repeats=3), claim="O(n^2)")
print()
growth_table(measure_growth(build_reversed_append, [400_000, 800_000, 1_600_000, 3_200_000],
                            repeats=3), claim="O(n)")

print()
print("The quadratic one is unmistakable: ratio ~4, and the fit says so.")
print()
print("The linear one shows a ratio of ~2. If the harness reports NOT SEPARABLE,")
print("that is O(n) and O(n log n) being indistinguishable at these sizes -- the")
print("situation NB-00 3.2 is about. The ratio is the evidence; the label is not.")
print()
print("Note the size columns: the quadratic version had to be measured at 5,000 to")
print("40,000, the linear one at 400,000 to 3,200,000 -- and the linear one is")
print("still faster in absolute terms at eighty times the size.")
print()
print("Three ways to recognise this in code you did not write:")
print("  - insert(0, ...) / pop(0) / del a[0] inside a loop")
print("  - `result = [x] + result` in a loop (allocates and copies every time)")
print("  - s += chunk in a loop over strings (NB-02 measures that one)")
print()
print("The fix is nearly always: append and reverse once, or use a deque.")

all three produce identical output.



         n        seconds      ratio
------------------------------------
     5,000       0.005181          -
    10,000       0.020316       3.92
    20,000       0.077915       3.84
    40,000       0.326437       4.19

best fit: O(n^2) (relative error 0.023); next: O(n log n) (1.134)
claimed O(n^2) -> measurement MATCHES the claim



         n        seconds      ratio
------------------------------------
   400,000       0.022062          -
   800,000       0.046379       2.10
 1,600,000       0.094885       2.05
 3,200,000       0.192977       2.03

best fit: O(n log n) (relative error 0.025); next: O(n) (0.034)
claimed O(n) -> measurement does NOT match the claim
  Do not paper over this. Either the claim is wrong, the input never
  reaches the worst case, or constant factors dominate at these sizes.

The quadratic one is unmistakable: ratio ~4, and the fit says so.

The linear one shows a ratio of ~2. If the harness reports NOT SEPARABLE,
that is O(n) and O(n log n) being indistinguishable at these sizes -- the
situation NB-00 3.2 is about. The ratio is the evidence; the label is not.

Note the size columns: the quadratic version had to be measured at 5,000 to
40,000, the linear one at 400,000 to 3,200,000 -- and the linear one is
still faster in absolute terms at eighty times the size.

Three ways to recognis

***
# Part 3 - The signature difficulty: cache locality

Everything so far treated memory as uniform, because that is what the cost model in NB-00 §1.2
assumes. It is the assumption that most misleads you about arrays specifically, because the whole
reason to use an array is a property the model cannot see.

A modern CPU does not fetch bytes; it fetches **cache lines**, typically 64 bytes. Touch one
`int` and you have paid for the 16 around it. Walk forward and the prefetcher notices and fetches
ahead of you. Jump around and you pay full price every time — and "full price" is roughly two
orders of magnitude more than a cache hit.

None of that appears in $\Theta(n)$. All of it appears in the wall clock.

## 3.1 Row-major versus column-major

A 2-D array is stored as one flat block. In **row-major** order (C, Java, numpy by default) `g[i][j]`
and `g[i][j+1]` are adjacent; `g[i][j]` and `g[i+1][j]` are a whole row apart.

So the two loop orders below perform **identical work** — same number of additions, same number of
index operations — and differ only in the order they visit memory.

In [18]:
# ---------------------------------------------------------------------------
# Java first: int[][] is a real 2-D layout, so the effect is not masked by
# interpreter overhead.
# ---------------------------------------------------------------------------
JAVA_GRID = """
public class GridScan {
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);
        int[][] g = new int[n][n];
        for (int i = 0; i < n; i++) for (int j = 0; j < n; j++) g[i][j] = 1;

        long s = 0;
        for (int w = 0; w < 3; w++) {                 // warm both paths
            for (int i = 0; i < n; i++) for (int j = 0; j < n; j++) s += g[i][j];
            for (int j = 0; j < n; j++) for (int i = 0; i < n; i++) s += g[i][j];
        }
        long t0 = System.nanoTime();
        for (int i = 0; i < n; i++) for (int j = 0; j < n; j++) s += g[i][j];
        double rowMajor = (System.nanoTime() - t0) / 1e9;

        t0 = System.nanoTime();
        for (int j = 0; j < n; j++) for (int i = 0; i < n; i++) s += g[i][j];
        double colMajor = (System.nanoTime() - t0) / 1e9;

        System.out.printf("%.6f %.6f %d%n", rowMajor, colMajor, s);
    }
}
"""

print("Java int[n][n], summing every element both ways:\n")
print("  %-16s %14s %16s %10s %14s"
      % ("array", "row-major (s)", "column-major (s)", "ratio", "size"))
print("  " + "-" * 76)
for n in [1000, 2000, 4000]:
    o = run_java(JAVA_GRID, args=[n]).split()
    rm, cm = float(o[0]), float(o[1])
    mb = n * n * 4 / 1024 / 1024
    print("  %-16s %14.4f %16.4f %9.2fx %11.0f MB"
          % ("%dx%d" % (n, n), rm, cm, cm / rm, mb))

print()
print("Identical operation counts. The only difference is the ORDER of access.")
print()
print("The largest array pays a heavy penalty: every column step lands on a fresh")
print("cache line, so 64 bytes are fetched to use 4 of them, and the prefetcher")
print("cannot help because the stride is a whole row.")
print()
print("But read the middle row before concluding the effect grows smoothly with")
print("size -- it does not, and on this machine the 2000x2000 case is barely")
print("penalised at all. What matters is not size in the abstract but whether the")
print("working set still fits in a cache level, which is a property of THIS")
print("machine. Practice 7 has you measure your own hierarchy and find the cliff.")

Java int[n][n], summing every element both ways:

  array             row-major (s) column-major (s)      ratio           size
  ----------------------------------------------------------------------------
  1000x1000                0.0026           0.0042      1.63x           4 MB


  2000x2000                0.0106           0.0107      1.00x          15 MB


  4000x4000                0.0169           0.1001      5.92x          61 MB

Identical operation counts. The only difference is the ORDER of access.

The largest array pays a heavy penalty: every column step lands on a fresh
cache line, so 64 bytes are fetched to use 4 of them, and the prefetcher
cannot help because the stride is a whole row.

But read the middle row before concluding the effect grows smoothly with
size -- it does not, and on this machine the 2000x2000 case is barely
penalised at all. What matters is not size in the abstract but whether the
working set still fits in a cache level, which is a property of THIS
machine. Practice 7 has you measure your own hierarchy and find the cliff.


In [19]:
# ---------------------------------------------------------------------------
# The same effect in Python, where it is smaller but still visible.
# ---------------------------------------------------------------------------
N = 1200
grid = [[1] * N for _ in range(N)]


def row_major(g):
    s = 0
    for row in g:
        for v in row:
            s += v
    return s


def col_major(g):
    s = 0
    for j in range(len(g)):
        for i in range(len(g)):
            s += g[i][j]
    return s


tr = min(measure_growth(row_major, [1], setup=lambda _n: grid, repeats=3)[0]["seconds"]
         for _ in range(1))
tc = min(measure_growth(col_major, [1], setup=lambda _n: grid, repeats=3)[0]["seconds"]
         for _ in range(1))
print("Python list-of-lists %dx%d:" % (N, N))
print("  row-major    %.4fs" % tr)
print("  column-major %.4fs   ratio %.2fx" % (tc, tc / tr))
print()
print("Read this one carefully, because it is NOT a clean measurement: col_major")
print("also does two index operations per element where row_major iterates rows")
print("directly, so interpreter overhead is mixed into the ratio.")
print()
print("A cleaner Python version -- identical index-op count, only the order differs:")

flat = [1] * (N * N)
seq = list(range(N * N))
strided = [j * N + i for j in range(N) for i in range(N)]


def sum_by(idx):
    s = 0
    for k in idx:
        s += flat[k]
    return s


ts = min(measure_growth(sum_by, [1], setup=lambda _n: seq, repeats=3)[0]["seconds"]
         for _ in range(1))
tst = min(measure_growth(sum_by, [1], setup=lambda _n: strided, repeats=3)[0]["seconds"]
          for _ in range(1))
print("  sequential   %.4fs" % ts)
print("  strided      %.4fs   ratio %.2fx" % (tst, tst / ts))
print()
print("Smaller than Java's, and that is the honest result: Python's per-element")
print("interpreter cost is large enough to hide much of the memory effect. The")
print("lesson is not 'Python has no cache' -- it is that the closer you get to the")
print("metal, the more layout dominates.")

Python list-of-lists 1200x1200:
  row-major    0.0415s
  column-major 0.0829s   ratio 2.00x

Read this one carefully, because it is NOT a clean measurement: col_major
also does two index operations per element where row_major iterates rows
directly, so interpreter overhead is mixed into the ratio.

A cleaner Python version -- identical index-op count, only the order differs:


  sequential   0.0604s
  strided      0.0654s   ratio 1.08x

Smaller than Java's, and that is the honest result: Python's per-element
interpreter cost is large enough to hide much of the memory effect. The
lesson is not 'Python has no cache' -- it is that the closer you get to the
metal, the more layout dominates.


## 3.2 Array of structs versus struct of arrays

The same principle at the level of your data model.

- **Array of structs (AoS)** — `Particle[] p`, each with `x, y, z`. Natural to write.
- **Struct of arrays (SoA)** — `double[] xs, ys, zs`. Awkward to write.

If you read *one field* of every record, AoS drags whole objects through cache and, in Java,
follows a reference per element to wherever the allocator put it. SoA reads a contiguous run of
exactly what you asked for.

In [20]:
# ---------------------------------------------------------------------------
# AoS vs SoA in Java, where objects genuinely live elsewhere on the heap.
# ---------------------------------------------------------------------------
JAVA_LAYOUT = """
public class Layout {
    static class Particle {
        double x, y, z;
        Particle(double v) { x = y = z = v; }
    }
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);
        Particle[] aos = new Particle[n];
        double[] xs = new double[n];
        for (int i = 0; i < n; i++) { aos[i] = new Particle(i); xs[i] = i; }

        double s = 0;
        for (int w = 0; w < 3; w++) {
            for (int i = 0; i < n; i++) s += aos[i].x;
            for (int i = 0; i < n; i++) s += xs[i];
        }
        long t0 = System.nanoTime();
        for (int i = 0; i < n; i++) s += aos[i].x;
        double tAos = (System.nanoTime() - t0) / 1e9;

        t0 = System.nanoTime();
        for (int i = 0; i < n; i++) s += xs[i];
        double tSoa = (System.nanoTime() - t0) / 1e9;

        System.out.printf("%.6f %.6f %.1f%n", tAos, tSoa, s);
    }
}
"""

print("summing ONE field of n records:\n")
print("  %-14s %18s %16s %10s" % ("n", "AoS Particle[] (s)", "SoA double[] (s)", "ratio"))
print("  " + "-" * 64)
for n in [5_000_000, 20_000_000]:
    o = run_java(JAVA_LAYOUT, args=[n]).split()
    ta, ts_ = float(o[0]), float(o[1])
    print("  %-14s %18.4f %16.4f %9.2fx" % ("{:,}".format(n), ta, ts_, ta / ts_))

print()
print("Same field, same count, same arithmetic. The SoA version reads a contiguous")
print("run of doubles; the AoS version dereferences a pointer per element and pulls")
print("in y and z it never uses.")
print()
print("When this matters: numerical work, graphics, simulation, columnar analytics.")
print("It is exactly why Parquet and every analytics database store data by COLUMN")
print("-- a query that reads two of forty columns should not pay for the other 38.")
print()
print("When it does not: if you touch every field of every record anyway, AoS is")
print("fine and far more readable. Measure before contorting your data model.")

summing ONE field of n records:

  n              AoS Particle[] (s) SoA double[] (s)      ratio
  ----------------------------------------------------------------


  5,000,000                  0.0123           0.0072      1.72x


  20,000,000                 0.0422           0.0196      2.16x

Same field, same count, same arithmetic. The SoA version reads a contiguous
run of doubles; the AoS version dereferences a pointer per element and pulls
in y and z it never uses.

When this matters: numerical work, graphics, simulation, columnar analytics.
It is exactly why Parquet and every analytics database store data by COLUMN
-- a query that reads two of forty columns should not pay for the other 38.

When it does not: if you touch every field of every record anyway, AoS is
fine and far more readable. Measure before contorting your data model.


## 3.3 Bulk operations, and when the intrinsic stops helping

§1.3's resize copies a whole buffer. Java offers `System.arraycopy` for that — a JVM *intrinsic*,
compiled to a specialised memory move rather than a loop, and skipping the per-element bounds
check.

So it should beat a hand-written loop. The interesting part is where it stops doing so.

In [21]:
# ---------------------------------------------------------------------------
# The same copy, two ways, across four orders of magnitude.
# ---------------------------------------------------------------------------
JAVA_COPY = """
public class CopyBench {
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);
        int[] src = new int[n];
        for (int i = 0; i < n; i++) src[i] = i;
        int[] a = new int[n], b = new int[n];

        for (int w = 0; w < 200; w++) {              // generous JIT warm-up
            for (int i = 0; i < n; i++) a[i] = src[i];
            System.arraycopy(src, 0, b, 0, n);
        }
        double manual = Double.MAX_VALUE, intrinsic = Double.MAX_VALUE;
        for (int r = 0; r < 20; r++) {               // minimum of 20 timed runs
            long t0 = System.nanoTime();
            for (int i = 0; i < n; i++) a[i] = src[i];
            manual = Math.min(manual, (System.nanoTime() - t0) / 1e9);

            t0 = System.nanoTime();
            System.arraycopy(src, 0, b, 0, n);
            intrinsic = Math.min(intrinsic, (System.nanoTime() - t0) / 1e9);
        }
        System.out.printf("%.8f %.8f %b%n", manual, intrinsic,
                          java.util.Arrays.equals(a, b));
    }
}
"""

print("copying an int[], minimum of 20 runs after 200 warm-up rounds:\n")
print("  %14s %16s %18s %10s" % ("n", "manual loop (s)", "arraycopy (s)", "speed-up"))
print("  " + "-" * 64)
for n in [1_000, 100_000, 5_000_000, 20_000_000]:
    o = run_java(JAVA_COPY, args=[n]).split()
    manual, intrinsic = float(o[0]), float(o[1])
    print("  %14s %16.8f %18.8f %9.2fx"
          % ("{:,}".format(n), manual, intrinsic, manual / intrinsic))
print("\n  (results identical in every case: %s)" % o[2])

print()
print("At small n the intrinsic wins by a wide margin: the loop's per-element")
print("bounds checks and loop overhead are the entire cost, and arraycopy skips")
print("them. Treat the smallest row as indicative rather than exact -- at that")
print("size the measurement is close to the timer's resolution.")
print()
print("At large n they converge. Neither is 'faster' any more, because both are")
print("limited by MEMORY BANDWIDTH: 20,000,000 ints is 80 MB read and 80 MB")
print("written, and no amount of instruction-level cleverness moves bytes quicker")
print("than the memory bus will carry them.")
print()
print("That is the general shape of intrinsics and vectorisation, and it is worth")
print("internalising: they remove per-element CPU overhead, so they help exactly")
print("until the bottleneck becomes the memory system instead.")

copying an int[], minimum of 20 runs after 200 warm-up rounds:

               n  manual loop (s)      arraycopy (s)   speed-up
  ----------------------------------------------------------------
           1,000       0.00000210         0.00000010     21.00x


         100,000       0.00001090         0.00001070      1.02x


       5,000,000       0.00245160         0.00202080      1.21x


      20,000,000       0.01188300         0.00923340      1.29x

  (results identical in every case: true)

At small n the intrinsic wins by a wide margin: the loop's per-element
bounds checks and loop overhead are the entire cost, and arraycopy skips
them. Treat the smallest row as indicative rather than exact -- at that
size the measurement is close to the timer's resolution.

At large n they converge. Neither is 'faster' any more, because both are
limited by MEMORY BANDWIDTH: 20,000,000 ints is 80 MB read and 80 MB
written, and no amount of instruction-level cleverness moves bytes quicker
than the memory bus will carry them.

That is the general shape of intrinsics and vectorisation, and it is worth
internalising: they remove per-element CPU overhead, so they help exactly
until the bottleneck becomes the memory system instead.


## 3.4 What this means in practice

Three rules that follow from the measurements above, in decreasing order of how often they matter:

1. **Iterate in memory order.** It is free. Getting the loop nesting right on a 2-D array costs
   nothing and bought several times the speed in §3.1.
2. **Prefer contiguous, primitive storage in hot paths.** `int[]` over `Integer[]`, `double[]` over
   `Double[]`, columnar over row-wise — once you have measured that the hot path is actually there.
3. **Do not contort your data model on suspicion.** SoA is harder to read and harder to change.
   §3.2's gap is real and it is a factor of two, not a hundred; a bad algorithm will cost you more
   than a good layout will save you.

And the general one, which is NB-00 §1.2 restated with evidence: **the cost model says memory
access is O(1), and that is the most consequential lie it tells.** Two implementations with
identical complexity can differ by a large constant, and on arrays that constant is usually about
layout.

***
# Part 4 - Tough questions

***

### Q1. Why is array indexing $O(1)$, and what does that cost you?

<details><summary>Answer</summary>

Because the address is **computed, not searched**:
$$ \text{addr}(a[i]) = \text{base} + i \times \text{sizeof(element)} $$
One multiply, one add, independent of $i$ and of $n$. §1.1 prints the actual addresses of an
`array.array` to make this concrete, and confirms that indexing `a[0]` and `a[n-1]` take the same
time.

**It costs two things**, and every array weakness follows from them:

1. **Equal-sized slots.** Either fixed-width values (Java `int[]`, Python `array.array`), or
   pointers to objects living elsewhere (Python `list`, Java `Integer[]`). §1.4 measures the
   indirection: `Integer[]` is **2.8×** slower to sum than `int[]`.
2. **One contiguous block.** So the size is fixed at allocation, growth means allocate-and-copy
   (§1.3), and inserting in the middle shifts everything after it (§1.2).

**The bonus nobody writes in the complexity:** contiguity means sequential traversal is
*prefetchable*. §3.1 measures the same Java array summed in the wrong order running **7.4× slower**
at 4000×4000. That is not in $\Theta(n)$ and it is often the largest term in practice.

</details>

***

### Q2. How does a dynamic array achieve amortised $O(1)$ append?

<details><summary>Answer</summary>

By growing **geometrically**. When the buffer fills, allocate `capacity × factor` (not
`capacity + k`), copy, and continue.

**Three proofs, same answer** (§1.3):

- **Aggregate.** With factor 2, $n$ appends copy $1+2+4+\dots < 2n$ elements total, so $O(n)$ for
  $n$ appends — $O(1)$ each.
- **Accounting.** Charge 3 units per append: 1 to store, 2 banked. A resize spends exactly what
  was banked since the last one. The bank never goes negative, so the amortised cost is the
  charge, $O(1)$.
- **Potential.** $\Phi = 2\cdot\text{size} - \text{capacity}$. Cheap appends raise it by 2, a
  resize consumes it, and actual + $\Delta\Phi$ stays constant.

**Verified, not asserted:** §1.3 measures $n$ appends at $O(n)$ total, and cross-checks the Python
and Java implementations to confirm they perform the *same* sequence of allocations — so the
argument transfers.

**The critical word is geometric.** Grow by a constant instead and NB-00 §1.5 measures the same
interface becoming $O(n)$ per append and $O(n^2)$ overall.

**Amortised is not average-case.** No adversary can make $n$ appends cost more than $O(n)$; the
argument does not depend on the values at all. A hash table's $O(1)$ is average-case and an
adversary *can* break it (NB-03).

</details>

***

### Q3. Why 2× and not 1.5×, or 4×? Does the growth factor matter?

<details><summary>Answer</summary>

It matters, and it is a genuine trade rather than a right answer. §1.3 sweeps it over 100,000
appends:

| factor | resizes | copies per append | final capacity | wasted |
|---|---|---|---|---|
| 1.125 | 91 | **8.24** | 102,908 | 2.9% |
| 1.5 | 29 | 2.77 | 138,255 | 38% |
| 2.0 | 17 | 1.31 | 131,072 | 31% |
| 4.0 | 9 | **0.87** | 262,144 | **162%** |

**Every row is amortised $O(1)$** — copies per append is a constant throughout. What changes is
*which* constant, and how much memory you hold that you are not using.

- **Small factor:** little waste, much more copying, and more allocator churn.
- **Large factor:** almost no copying, and a buffer that can be more than half empty.

**What the real implementations chose:** CPython's `list` grows by roughly **1.125×**, Java's
`ArrayList` by **1.5×**. C++ implementations differ — libstdc++ uses 2, MSVC uses 1.5.

**The argument for factors below 2** is memory reuse: with factor 2, the new block is always
larger than the sum of all previously freed blocks, so a simple allocator can never reuse them.
With a factor below the golden ratio (~1.618), freed blocks eventually coalesce into something big
enough. Whether that matters depends entirely on your allocator — which is why the answer to
"what is the best factor" is genuinely "it depends", and why implementations disagree.

</details>

***

### Q4. When would you *not* use an array?

<details><summary>Answer</summary>

Four cases, all following from §1.2:

1. **Frequent insertion or deletion at the front or middle.** Each is $\Theta(n)$. §2.4 measures
   `insert(0, x)` in a loop at a clean $O(n^2)$. Use a **deque** (NB-05) for the ends, a **linked
   list** (NB-04) if you already hold a reference to the position.
2. **You need ordering-based queries** — predecessor, successor, range — on data that keeps
   changing. A sorted array is $\Theta(n)$ to update; a **balanced BST** (NB-08) is $\Theta(\log n)$.
3. **Membership testing is the main operation.** $\Theta(n)$ scan versus a **hash set**'s $\Theta(1)$
   (NB-03). `if x in some_list` inside a loop is the second most common quadratic accident.
4. **Sparse data.** A million slots for a thousand values wastes memory; use a dict or a sparse
   representation.

**And the anti-case, which is more common than any of the above:** people abandon arrays for
linked lists on the strength of "$O(1)$ insertion" and lose. NB-04 measures traversal of a linked
list against an array of the same length; the constant from cache behaviour (§3) is large enough
to swamp the complexity advantage for most realistic workloads.

</details>

***

### Q5. Explain prefix sums, and when they stop being the right answer.

<details><summary>Answer</summary>

Precompute $P[k] = \sum_{i<k} a[i]$ with $P[0]=0$, then any range sum is one subtraction:
$$ \sum_{i=lo}^{hi-1} a[i] = P[hi] - P[lo] $$
Build $\Theta(n)$ once, query $\Theta(1)$ forever. §2.1 verifies it against brute force over
5,000 random ranges including the empty range and both endpoints — where the off-by-one bugs are.

**It works because the array gives $O(1)$ access to both endpoints.** The identity is equally true
for a linked list and equally useless there.

**When it stops being right:**

- **The array changes.** A single update invalidates every prefix from that point on — $\Theta(n)$
  to rebuild. If you need updates *and* range queries, you want a **Fenwick tree or segment tree**
  (NB-12), which is $\Theta(\log n)$ for both.
- **The operation is not invertible.** The trick is $P[hi] - P[lo]$; it needs subtraction. Sums,
  XORs and counts work. Minimum and maximum do not — you cannot "subtract" a minimum — so range
  minimum needs a sparse table or a segment tree.
- **Overflow.** Prefix sums grow with $n$. In Java, prefix sums of `int` need a `long` accumulator;
  Python's unbounded integers hide the bug until you port the code (NB-00 §2.3).

**The mirror image is the difference array** (§2.2): prefix sums make range *queries* $O(1)$ on a
static array, difference arrays make range *updates* $O(1)$ when you only read at the end.
§2.2 measures the second at **391× faster** than the naive version at 400 updates.

</details>

***

### Q6. How does the two-pointer technique work, and why is it linear?

<details><summary>Answer</summary>

**The idea:** when the array has structure you can exploit — usually sortedness — you can move an
index and know you will never need to move it back.

**Opposite ends, on a sorted array**, looking for a pair summing to `target`:
start `lo=0`, `hi=n-1`. If `a[lo]+a[hi] < target`, then `a[lo]` is too small to pair with *any*
remaining element, so `lo++` discards an entire row of the $n^2$ candidate pairs. If it is too
large, `hi--` discards a column.

**Why linear:** each step eliminates at least one index permanently, and there are $n$ indices. So
at most $n$ steps — checked in §2.3 on the worst case where no pair exists.

**Sliding window** is the same argument with both pointers moving forward: extend `hi` to include
elements, advance `lo` to restore the invariant. It *looks* quadratic (a `while` inside a `for`),
and it is not, because **each pointer advances at most $n$ times across the entire run** — total
work $\le 2n$. That is an amortised argument (NB-00 §1.5), and it is the part people get wrong
when asked to justify the complexity.

**The precondition matters.** Two pointers on a sorted array needs sortedness — if you must sort
first, you have paid $\Theta(n\log n)$ and a hash-based $\Theta(n)$ solution may be better.
Sliding window needs the property to be **monotone** in the window: extending can only push it one
way. "Longest subarray with sum ≤ k" works for non-negative values; introduce negatives and
extending can decrease the sum, the invariant breaks, and the technique silently returns wrong
answers.

</details>

***

### Q7. Your service got slow after the input doubled. It uses lists heavily. Where do you look?

<details><summary>Answer</summary>

Get the ratio first (NB-00 Q11): ~2 is linear growth, **~4 means something is quadratic**. If it
is ~4, the array-specific suspects, in order of frequency:

1. **`insert(0, x)` / `pop(0)` / `del a[0]` in a loop.** Each is $\Theta(n)$ (§1.2). §2.4 measures
   this at a textbook $O(n^2)$. Fix: `append` and `reverse` once, or `collections.deque`.
2. **`if x in some_list` inside a loop.** $\Theta(n)$ per test. Fix: build a `set` once.
3. **`result = result + [x]` or `result += [x]` in a loop** — allocates and copies the whole list
   each time. Fix: `append`.
4. **String concatenation in a loop** — the same bug on a different type (NB-02).
5. **Slicing in a loop.** `a[1:]` copies. A recursive function doing `f(a[1:])` is $\Theta(n^2)$
   even though it looks $\Theta(n)$. Fix: pass an index.

**If the ratio is ~2 but everything is slower**, suspect memory rather than algorithms: the working
set may have outgrown a cache level. §3.1 shows the same operation getting **7.4×** worse purely
from array size crossing that threshold, with no code change at all.

**Then profile** — to confirm, not to explore. Profiling first tells you where time goes without
telling you why it grew.

</details>

***

### Q8. `int[]` versus `Integer[]` versus `List<Integer>` in Java — does it matter?

<details><summary>Answer</summary>

Yes, and §1.4 measures it: summing 20,000,000 elements, `Integer[]` is **2.8× slower** than `int[]`.

**Memory**, which drives the time difference:

| | Per element |
|---|---|
| `int[]` | 4 bytes, contiguous |
| `Integer[]` | 8-byte reference **+ ~16-byte object** on the heap |

So `Integer[]` is roughly **6× the memory**, and each element costs a pointer dereference to
wherever the allocator put the object — which is exactly the AoS problem from §3.2.

**`List<Integer>` adds more:** `ArrayList` is backed by an `Object[]`, so you get the boxing *plus*
the collection's own overhead, plus autoboxing on every `add`/`get` if you are not careful.
Java has no `List<int>` — generics do not accept primitives.

**In practice:**

- **Hot numeric paths:** use `int[]`, `double[]`, `long[]`. This is why `IntStream` exists
  separately from `Stream<Integer>`.
- **Everything else:** `List<Integer>` is fine and clearer. This is a hot-path optimisation, not a
  style rule.
- **If you need both**, primitive-collection libraries (fastutil, Trove, Eclipse Collections)
  provide `IntArrayList` and friends.

**The gotcha worth knowing:** Java caches `Integer` objects for −128..127, so `==` on small boxed
values appears to work and then fails at 128. Always `.equals()` — or better, do not box.

</details>

***

### Q9. What is the difference between capacity and size, and why is it visible?

<details><summary>Answer</summary>

**Size** is how many elements you have. **Capacity** is how many the buffer can hold before it must
grow. The invariant is `0 <= size <= capacity` — §1.3 asserts exactly that after every one of
20,000 random operations.

**Why it leaks into your life:**

- **Memory.** A list holding 100,000 items may own a buffer for 162,000 (§1.3, factor 4). If you
  keep many such lists, that is real. Java's `ArrayList.trimToSize()` exists for this; in Python,
  `list(x)` produces a right-sized copy.
- **Pre-sizing avoids all the copying.** `new ArrayList<>(expectedSize)` or
  `[None] * n` skips every resize. When you know the size, say so.
- **Removing elements does not free memory.** Python lists shrink the buffer only when they drop
  well below half full, and Java's `ArrayList` never shrinks on its own.
- **Iterator invalidation.** In Java, structurally modifying a list while iterating throws
  `ConcurrentModificationException`; in C++ a `vector` reallocation invalidates every existing
  pointer and iterator — a genuine source of undefined behaviour.

**The thing people get wrong:** `sys.getsizeof(a_list)` reports the *pointer buffer only*, not the
objects. §1.4 shows the honest accounting — 36 bytes per element for a list of ints, against 4.2
for `array.array('i')`, an **8.5× difference** that the naive measurement reports as none.

</details>

***

### Q10. When is `array.array` or numpy worth it over a plain list?

<details><summary>Answer</summary>

§1.4 measures both sides, and the result is not one-directional:

| | list of ints | `array.array('i')` |
|---|---|---|
| memory, 2M ints | 72 MB (36 B/elem) | 8.5 MB (4.2 B/elem) |
| `sum()` | **faster** | **2.6× slower** |

**The memory win is large and the speed result is backwards from what people expect.** The reason
is boxing: `array.array` stores raw machine ints, so every element Python touches must be wrapped
in a fresh `int` object first. A list already holds objects, so iterating it only follows pointers.

**So use `array.array` when** the data is large and something *other than a Python loop* consumes
it — writing to a file or socket, `memoryview`, or handing the buffer to a C library. The whole
point is that the bytes are already in the right layout.

**Use numpy when** you can express the operation as a **whole-array** operation. `arr.sum()` runs
in C over contiguous memory and beats both — but `for x in arr` in Python is *slower* than a list,
for exactly the boxing reason above. numpy's win comes from not looping in Python at all.

**Stay with a list when** the data is small, heterogeneous, or you are doing element-wise Python
work on it. A list is the right default; these are the exceptions, and each needs measuring.

</details>

***

### Q11. Two implementations have the same $O(n)$ complexity. One is 5× faster. What are the candidates?

<details><summary>Answer</summary>

For arrays specifically, in rough order of how often it is the answer:

1. **Memory access order.** §3.1: the same Java array summed in the wrong loop order is **7.4×**
   slower at 4000×4000. Same operations, same count.
2. **Data layout.** §3.2: reading one field of 20,000,000 records is **2.45×** slower from an array
   of objects than from a parallel array of primitives.
3. **Boxing / indirection.** §1.4: `Integer[]` is **2.8×** slower than `int[]`; every element is a
   pointer to somewhere else.
4. **Hidden copying.** A slice, a `+`, or a defensive copy inside the loop. This one often turns
   out not to be $O(n)$ at all once you look (§2.4).
5. **Language and runtime.** NB-00 §1.3 measured the identical loop hundreds of times apart between
   Python and Java.
6. **What the compiler did.** JIT, bounds-check elimination, vectorisation. §2 of this notebook
   has an example: `System.arraycopy` is **25× faster** than a manual copy loop at n=1,000 and
   roughly a **wash at 20,000,000**, because at that size both are limited by memory bandwidth.

**The diagnostic order:** confirm the complexity really is the same (measure, do not read); then
look at access order; then at layout; then profile for hidden allocation. Do not start by
micro-optimising arithmetic — it is almost never the answer.

</details>

***

### Q12. What is a jagged array, and why do 2-D arrays differ between languages?

<details><summary>Answer</summary>

There are two genuinely different things people call a 2-D array:

**True 2-D (contiguous).** One block of $rows \times cols$ elements; `g[i][j]` is at
`base + (i*cols + j)*size`. One allocation, one address computation, perfectly prefetchable.
C's `int g[10][20]`, numpy's `ndarray`, and Java's `int[][]` when you allocate it rectangularly.

**Jagged (array of arrays).** An array of *references*, each to a separate row object, each
allocated independently and possibly of different lengths. `g[i][j]` is two dereferences, and the
rows can be anywhere in memory. This is what Java's `int[][]` and Python's list-of-lists actually
are.

| | Layout | `g[i][j]` |
|---|---|---|
| C / numpy | one contiguous block | one address computation |
| Java `int[][]` | array of row references | two dereferences |
| Python list-of-lists | list of list objects | two dereferences, both boxed |

**Consequences:**

- **Rows need not be the same length**, which is occasionally useful and mostly a footgun.
- **Row-major locality holds *within* a row** and not across rows, so §3.1's effect is about
  scanning each row contiguously rather than about one big block.
- **Memory overhead:** an $n \times n$ jagged array carries $n$ separate objects with their own
  headers, plus the reference array.
- **In Python, `[[0] * n] * n` is a classic bug** — it makes $n$ references to *the same* row, so
  writing `g[0][0]` changes every row. Use `[[0] * n for _ in range(n)]`.

**If you need real 2-D numerics in Python, use numpy** — it gives you the contiguous block, the
single allocation, and the whole-array operations that avoid Python-level iteration entirely.

</details>

***

## Coding challenges

### Challenge 1 — a dynamic array with shrinking

§1.3 built one that only grows. Add `pop` that shrinks the buffer, and find the rule that keeps it
amortised $O(1)$.

1. Shrink to half capacity when size falls below **half**. Now construct the sequence that makes
   this $\Theta(n)$ *per operation*: alternate push/pop at the boundary so every operation
   reallocates. Measure it and confirm the ratio.
2. Fix it by shrinking at **one quarter** full instead. Explain, using the potential method, why
   the gap between the grow and shrink thresholds is what restores the amortised bound.
3. Verify with a randomised stress test: 100,000 random push/pop operations, asserting the
   invariant after each, and confirming the total copies stay $O(n)$.
4. What does CPython actually do? Instrument `sys.getsizeof` across a long sequence of `append`
   and `pop` and find the real shrink threshold.

***

### Challenge 2 — 2-D locality, properly

§3.1 measured row-major versus column-major. Go further.

1. Implement naive matrix multiply (three nested loops) for $n = 512$. Time all six loop
   orderings. They compute the same result; the spread will be large. Explain the winner and the
   loser from the access patterns.
2. Implement **tiled** (blocked) multiplication and sweep the tile size. Find the value that
   performs best on your machine, then work out what cache size that implies.
3. Do it in both languages. Does the best tile size agree? Should it?
4. Compare against numpy's `@` and report how far off you are. The gap is BLAS, and knowing its
   size is the point of the exercise.

***

### Challenge 3 — find the quadratic

§2.4 lists the ways arrays go quadratic by accident. Build the detector.

1. Write five functions that each look linear and are quadratic, one per mechanism in §2.4.
2. Write a harness that runs any function at doubling sizes and reports "suspect quadratic" when
   the ratio is near 4, using `fit_complexity` from `dsa_toolkit`.
3. Make it robust to the failure modes NB-00 §3.2 identified: refuse to answer when the sizes are
   too small or the range too narrow.
4. Run it over a real codebase's hot functions — yours, or an open-source project. Report what you
   find, including the false positives, and what distinguished them.

***
# Part 5 - Practice

| # | Exercise | Skill it forces | Difficulty |
|---|---|---|---|
| 1 | Rotate an array in place | Index arithmetic, no extra space | ★☆☆☆☆ |
| 2 | Running maximum with prefix logic | When prefix sums do *not* apply | ★★☆☆☆ |
| 3 | 2-D prefix sums | Extending §2.1 to two dimensions | ★★★☆☆ |
| 4 | Sliding window with a twist | Recognising when the invariant breaks | ★★★☆☆ |
| 5 | Implement `ArrayList` faithfully | Capacity, shrinking, iterator invalidation | ★★★☆☆ |
| 6 | Dutch national flag | Three-way partition in one pass | ★★★☆☆ |
| 7 | Measure your cache | Turning §3 into a number | ★★★★☆ |
| 8 | Sparse array | When contiguity is the wrong answer | ★★★★☆ |

***

### 1. Rotate an array in place

Rotate `a` left by `k` positions using $O(1)$ extra space.

1. Do it with the reversal trick: reverse `a[0:k]`, reverse `a[k:n]`, reverse the whole thing.
   Convince yourself why that works before coding it.
2. Handle `k > n`, `k == 0`, and the empty array. Stress-test against a slicing reference.
3. Now do it with the cyclic-replacement method. It is $O(n)$ with genuinely $O(1)$ space and
   involves a `gcd`. Which is faster, and why is it not the one with better constants on paper?

**The trap:** the naive "rotate by one, k times" is $O(nk)$, which is quadratic when $k \sim n$.

***

### 2. Running maximum, and where prefix logic fails

Given an array, answer many "maximum of `a[lo:hi]`" queries.

1. Try to build a prefix-max array the way §2.1 builds prefix sums. Show precisely why
   `M[hi] - M[lo]` is meaningless here.
2. Conclude what property the prefix trick actually requires. State it precisely.
3. Implement a **sparse table** ($O(n\log n)$ build, $O(1)$ query) and verify against brute force.
4. Which operations admit the prefix trick? Test sum, XOR, product, min, gcd — and say which fail
   and why.

***

### 3. 2-D prefix sums

Extend §2.1 to rectangles: sum of the submatrix from $(r_1,c_1)$ to $(r_2,c_2)$ in $O(1)$.

1. Derive the inclusion-exclusion formula. Draw it before writing it.
2. Implement it with the same `+1` padding convention §2.1 uses, and stress-test against brute
   force over thousands of random rectangles including empty ones.
3. Build for a 2000×2000 grid and time 100,000 queries against the naive version.
4. Extend to a difference-array version for 2-D range *updates* (§2.2).

***

### 4. Sliding window with a twist

§2.3 warned that sliding windows need a monotone invariant.

1. Implement "longest subarray with sum ≤ k" for non-negative values, and verify it.
2. Now allow negative values. Find an input where it returns the wrong answer, and explain which
   step of the argument fails.
3. Solve the negative-value version correctly with prefix sums plus a monotonic deque, and
   verify it against brute force.
4. Write down the test you would add to a code review to catch this class of bug.

***

### 5. Implement `ArrayList` faithfully

1. Implement `add`, `get`, `set`, `remove(index)`, `size`, with the invariant asserted after every
   operation.
2. Match Java's actual growth: `newCapacity = oldCapacity + (oldCapacity >> 1)`. Compare the copy
   counts against §1.3's table.
3. Add a `modCount` and make your iterator throw on structural modification, the way Java's does.
   Then write the test that proves it.
4. Add `trimToSize` and `ensureCapacity`, and measure what pre-sizing saves for a known-length
   build.

***

### 6. Dutch national flag

Sort an array of 0s, 1s and 2s in a single pass with $O(1)$ space.

1. Implement it with three pointers. State the loop invariant explicitly before you code it —
   this problem is *about* the invariant.
2. Assert the invariant inside the loop with `check_invariant` and stress-test the result.
3. Prove each element is examined at most once, then confirm by counting.
4. Generalise to $k$ values. At what $k$ does counting sort (NB-14) become the better answer?

***

### 7. Measure your own cache

Turn §3 into a number.

1. Allocate arrays of increasing size and time a strided traversal of each. Plot time per access
   against array size.
2. The plateaus are your cache levels. Read L1, L2 and L3 sizes off your own graph.
3. Sweep the stride at a fixed size. The step where cost jumps is your cache-line size — you
   should find 64 bytes.
4. Check against your CPU's published specification. Being able to measure this is worth more than
   remembering the numbers.

***

### 8. Sparse arrays

An array where 99.9% of a million entries are zero.

1. Measure the memory of a plain list, `array.array`, a dict, and a list of (index, value) pairs.
2. Compare access time for each. Note where the dict's $O(1)$ loses to the array's contiguity
   despite matching complexity.
3. Find the density at which the dense representation becomes the better choice — by measuring,
   not guessing.
4. Implement compressed sparse row (CSR) for a 2-D sparse matrix and explain why it is built from
   three flat arrays rather than a dict.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapter 17.4 — "Dynamic tables".**
> The amortised analysis of exactly the structure §1.3 builds, done all three ways. Short, and the
> potential-method treatment is the one that makes the growth-factor trade-off feel inevitable
> rather than arbitrary.

**2. *What Every Programmer Should Know About Memory* — Ulrich Drepper, 2007.**
> The definitive treatment of everything in §3, and freely available. Long, and you do not need
> all of it: sections 3 (CPU caches) and 6 (what programmers can do) are the parts that turn §3.1's
> and §3.2's measurements into a mental model you can apply. Written for C programmers, and the
> hardware has not changed in the ways that matter.

**3. *Programming Pearls*, Jon Bentley — columns 1 and 2.**
> Column 1 is a sorting problem solved with a bit array under a hard memory limit, and it is the
> best short argument that representation *is* the algorithm. Column 2 covers the rotation problem
> in Practice 1, including the reversal trick and why it beats the obvious approach.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — contiguity and address arithmetic | **CLRS** ch. 10.1; **Sedgewick & Wayne**, *Algorithms* §1.3 | 🔍 |
| 1.3 — **dynamic arrays, amortised** | **CLRS** ch. 17.4. **Tarjan**, *Amortized computational complexity*, SIAM J. Alg. Disc. Meth. **1985** | 🔍 |
| 1.3 — the growth factor in practice | **CPython** `Objects/listobject.c`, `list_resize` — [github.com/python/cpython](https://github.com/python/cpython/blob/main/Objects/listobject.c); the JDK's `ArrayList.grow` | ✅ |
| 1.4 — boxing and object layout | **Shipilëv**, *Java Objects Inside Out* — [shipilev.net](https://shipilev.net/jvm/objects-inside-out/) | ✅ |
| 2.1–2.2 — prefix sums, difference arrays | **Bentley**, *Programming Pearls* col. 8 (the maximum-subarray development); competitive-programming folklore otherwise | 🔍 |
| 2.3 — two pointers, sliding window | **Sedgewick & Wayne** §1.4 for the amortised argument; **Bentley** col. 2 | 🔍 |
| 3.1–3.2 — **cache behaviour** | **Drepper**, *What Every Programmer Should Know About Memory*, **2007** — [lwn.net/Articles/250967](https://lwn.net/Articles/250967/) | ✅ |
| 3.1 — the numbers behind it | **Hennessy & Patterson**, *Computer Architecture: A Quantitative Approach*, ch. 2 | 🔍 |
| 3.2 — AoS/SoA and columnar storage | **Abadi, Madden & Hachem**, *Column-Stores vs. Row-Stores*, SIGMOD **2008** | 🔍 |
| Q12 — 2-D layout and numpy | **NumPy** internals documentation on strides and memory layout — [numpy.org](https://numpy.org/doc/stable/reference/arrays.ndarray.html) | ✅ |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Drepper's memory paper, sections 3 and 6.** CLRS ch. 17.4 is the more elegant piece of
mathematics, and you can absorb the amortised argument from §1.3 without it. Drepper is the one
that changes how you write code, because §3's measurements — 7.4× from loop order, 2.45× from data
layout — are not anomalies you can look up. They are the normal behaviour of every machine you
will deploy on, and they are invisible in every complexity analysis you will ever write.

Read it against §3.1 and §3.2, then do Practice 7 and measure your own cache hierarchy. Numbers you
measured yourself are the ones you remember.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Loop got quadratic and looks linear | `insert(0,·)`, `pop(0)`, `del a[0]` (§2.4) | `append` + `reverse`, or a `deque` (NB-05) |
| `x in a_list` inside a loop | $\Theta(n)$ membership test | Build a `set` once (NB-03) |
| `result = result + [x]` in a loop | Allocates and copies each time (§2.4) | `append` |
| Recursion on `a[1:]` is quadratic | Slicing copies (§2.4) | Pass an index, not a slice |
| `[[0]*n]*n` rows all change together | $n$ references to one row (Q12) | `[[0]*n for _ in range(n)]` |
| Memory much higher than expected | `list` is pointers + int objects (§1.4) | `array.array`, numpy, or `__slots__` |
| `sys.getsizeof` says the list is small | It excludes referenced objects (§1.4) | Sum the elements too, or `tracemalloc` |
| `array.array` is *slower* than a list | Boxing on every Python-level access (§1.4) | Correct and expected — use it for bulk I/O, not loops |
| Java `==` on boxed ints fails above 127 | `Integer` cache is −128..127 (Q8) | `.equals()`, or use `int` |
| Nested loop far slower one way round | Column-major access (§3.1) | Iterate in memory order |
| Reading one field of many objects is slow | AoS drags whole objects through cache (§3.2) | Parallel arrays, if measurement justifies it |
| Prefix sums give wrong range totals | Off-by-one in the `+1` convention (§2.1) | `P[hi] - P[lo]`, half-open, `len(P) == n+1` |
| Prefix sums overflow in Java | `int` accumulator over many elements | `long`, and cross-check against Python (NB-00 §2.3) |
| Sliding window wrong with negatives | The invariant is not monotone (§2.3) | Prefix sums + monotonic deque |
| List holds memory after removals | Buffers shrink late or never (Q9) | `list(x)` to right-size; `trimToSize()` in Java |

## Checklist for array code

- [ ] Is any $\Theta(n)$ operation — `insert(0)`, `pop(0)`, `in`, a slice — inside a loop (§2.4)?
- [ ] If the final size is known, is the array **pre-sized** (Q9)?
- [ ] For many range queries on static data, are **prefix sums** used (§2.1)?
- [ ] For many range updates read once at the end, is a **difference array** used (§2.2)?
- [ ] If a sliding window is used, is its invariant genuinely **monotone** (§2.3)?
- [ ] Are nested loops iterating in **memory order** (§3.1)?
- [ ] In a hot numeric path, is the storage **primitive and contiguous** (§1.4)?
- [ ] Has memory been measured **including the referenced objects** (§1.4)?
- [ ] In Java, could any accumulator **overflow `int`** (Q5)?
- [ ] Is the complexity claim **measured**, not assumed (NB-00 §1.4)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `strings_zero_to_hero.ipynb` | Strings are arrays with immutability bolted on — and §2.4's quadratic accident has a famous string version |
| `linked_lists_zero_to_hero.ipynb` | The structure that trades §1.1's contiguity away, and §3's measurements are why that trade usually loses |
| `stacks_queues_zero_to_hero.ipynb` | The deque that fixes §1.2's front-insertion problem, and the circular buffer that makes it work |
| `hashing_zero_to_hero.ipynb` | The $\Theta(1)$ membership test that §2.4 and Q7 keep pointing at |
| [`complexity_zero_to_hero.ipynb`](complexity_zero_to_hero.ipynb) | The amortised analysis §1.3 leans on, and the measurement discipline used throughout |

See [`README.md`](README.md) for the full roster and reading order.